#### Description of the Data

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import scienceplots
import os 

plt.style.use(['science', 'nature','bright'])
mpl.rcParams['savefig.format'] = 'svg'
os.makedirs('plots/efus2017/london_bedroom_4month/description', exist_ok=True)

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

ROOT = os.path.abspath('.')

# Extract room name for later
path = f"{ROOT}/efus_indoor_outdoor_bedroom.parquet"
df = pd.read_parquet(path)
room_name = str(path.split("/")[-1].replace(".parquet", "").split("_")[-1])

# Load parquet file 
print("Loading temperature data for",room_name,"...")
df = pd.read_parquet(f"{ROOT}/efus_indoor_outdoor_bedroom.parquet")

# Filter out extreme values
df = df[df["T_in"].between(-10, 40) & df["T_out"].between(-10, 40)]

# Add building characteristic data
BLDG_COLS = ["CaseID","dwtype_efus","dwage_efus","WallType2x_efus",
             "InsulatedWalls_efus","FullyDblGlz_efus","floor6x_efus","EPceeb12e_efus",
             "gorEHS_efus","AnyCooling"]
bldg = pd.read_csv(
    f"{ROOT}/test_data/ukda_9434_csv_r/csv/selected_interview_responses_caseid.csv",
    usecols=BLDG_COLS)
df = df.merge(bldg, on="CaseID", how="left").dropna(subset=BLDG_COLS[1:])

# Nominal
for c in BLDG_COLS[1:]:
    df[c] = df[c].astype(int)

# Filter to London subset 
df = df[df["gorEHS_efus"] == 7].copy()
print(f"London: {df['CaseID'].nunique()} dwellings, {len(df):,} observations")

# Filter to May-September
df = df[df['hour'].dt.month.between(5, 9)]

# Derived features
T_OUT_MEAN = df["T_out"].mean()
T_OUT_SD   = df["T_out"].std()

# Centered T_out on global mean across all dwellings and times
df["T_out_c"]     = df["T_out"] - T_OUT_MEAN
df["hour_of_day"] = df["hour"].dt.hour

# Descriptive stats 
df["T_out_q95"] = df['T_out'].quantile(0.95)
df["T_out_q5"] = df['T_out'].quantile(0.05)
median = df["T_out"].median()
q05 = df["T_out"].quantile(0.05)
q25 = df["T_out"].quantile(0.25)
q75 = df["T_out"].quantile(0.75)
q95 = df["T_out"].quantile(0.95)
iqr = q75 - q25

# Population-level diurnal terms
df["sin_h"] = np.sin(2 * np.pi * df["hour_of_day"] / 24)
df["cos_h"] = np.cos(2 * np.pi * df["hour_of_day"] / 24)

# At least 20 measurements over the measurement period
sizes = df.groupby("CaseID").size()
df_m = df[df["CaseID"].isin(sizes[sizes >= 20].index)].reset_index(drop=True)
df_m["dwelling"] = df_m["CaseID"].astype(str)

n_obs = len(df_m)
n_dwells = df_m['dwelling'].nunique()
r = int(n_obs/n_dwells/7)

print(f"\nModel dataset           : {n_obs} observations, {n_dwells} dwellings ({r} obs/dwells/week)")
print(f"T_out                   : median = {median:.1f}°C [IQR {iqr:.1f}°C]")
print(f"T_out                   : mean = {T_OUT_MEAN:.2f}°C,  SD = {T_OUT_SD:.2f}°C")
print(f"T_out central 90% range : {q05:.1f}–{q95:.1f}°C")

df.describe()

In [ ]:
# Building characteristics 

# One row per dwelling
dwelling_chars = (
    df_m[[
        "CaseID",
        "dwtype_efus",
        "dwage_efus",
        "WallType2x_efus",
        "InsulatedWalls_efus",
        "FullyDblGlz_efus",
        "floor6x_efus",
        "EPceeb12e_efus",
        "AnyCooling"
    ]]
    .drop_duplicates()
)

# Labels from efus
LEVELS = {

    "dwtype_efus": {
        1: "Detached",
        2: "Semi-detached",
        3: "End-terrace",
        4: "Mid-terrace",
        5: "Bungalow",
        6: "Flat/maisonette"
    },

    "dwage_efus": {
        1: "pre-1919",
        2: "1919–1944",
        3: "1945–1964",
        4: "1965–1974",
        5: "1975–1980",
        6: "1981–1990",
        7: "post-1990"
    },

    "WallType2x_efus": {
        1: "Solid wall",
        2: "Cavity wall"
    },

    "InsulatedWalls_efus": {
        0: "No",
        1: "Yes"
    },

    "FullyDblGlz_efus": {
        0: "No",
        1: "Yes"
    },

    "floor6x_efus": {
        1: r"$<50$ sqm",
        2: "50 - 69 sqm",
        3: "70 - 89 sqm",
        4: "90 - 109 sqm",
        5: "110 - 139 sqm",
        6: r"$>140$ sqm"
    },

    "EPceeb12e_efus": {
        1: "$>$ 70 (C+)",
        2: "30 - 50 (D)",
        3: "51 - 70 (E)",
        4: "$<$ 30 (F/G)"
    },

    "AnyCooling": {
        0: "No",
        1: "Yes"
    }
}

print(f"\nBuilding characteristics ({len(dwelling_chars)} dwellings)\n")

for col in LEVELS.keys():

    counts = dwelling_chars[col].value_counts().sort_index()
    perc = 100 * counts / counts.sum()

    summary = pd.DataFrame({
        "code": counts.index,
        "level": [LEVELS[col].get(i, "Unknown") for i in counts.index],
        "count": counts.values,
        "percent": perc.values.round(1)
    })

    print(f"\n{col}")
    print(summary.to_string(index=False))

# Most common archetype 
archetype_cols = [
    "dwtype_efus",
    "dwage_efus",
    "WallType2x_efus",
    "InsulatedWalls_efus",
    "FullyDblGlz_efus",
    "floor6x_efus",
    "EPceeb12e_efus",
    "AnyCooling"
]

archetypes = (
    dwelling_chars[archetype_cols]
    .value_counts()
    .reset_index(name="count")
)

top = archetypes.iloc[0]
print("\nMost common dwelling archetype\n")

for col in archetype_cols:
    label = LEVELS[col].get(top[col], str(top[col]))
    print(f"{col:22s}: {label}")

print(f"\nCount: {top['count']} dwellings")
print("\nMost common characteristics\n")

for col in archetype_cols:
    # Most frequent code
    mode_code = dwelling_chars[col].mode()[0]
    # Count
    count = (dwelling_chars[col] == mode_code).sum()
    # Percentage
    percent = 100 * count / len(dwelling_chars)
    # Label
    label = LEVELS[col].get(mode_code, str(mode_code))
    print(
        f"{col:22s}: {label:20s} "
        f"({count} dwellings, {percent:.1f}%)"
    )

In [ ]:
import matplotlib.dates as mdates
import textwrap

def plot_temps(dwelling_ids):

    # Ensure iterable
    if isinstance(dwelling_ids, str):
        dwelling_ids = [dwelling_ids]

    n = len(dwelling_ids)
    ncols = 2
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(12, 4.8 * nrows),
        squeeze=False
    )

    axes = axes.flatten()

    for ax, random_dwelling in zip(axes, dwelling_ids):

        # Subset data for that dwelling
        df_plot = (
            df_m[df_m["dwelling"] == str(random_dwelling)]
            .sort_values("hour")
        )

        # Building characteristics
        row = df_plot.iloc[0]

        char_text = (
            f"{LEVELS['dwtype_efus'][row['dwtype_efus']]}, "
            f"{LEVELS['dwage_efus'][row['dwage_efus']]}, "
            f"{LEVELS['WallType2x_efus'][row['WallType2x_efus']]}, "
            f"Insulated walls: "
            f"{LEVELS['InsulatedWalls_efus'][row['InsulatedWalls_efus']]}, "
            f"Double glazing: "
            f"{LEVELS['FullyDblGlz_efus'][row['FullyDblGlz_efus']]}, "
            f"Floor: {LEVELS['floor6x_efus'][row['floor6x_efus']]}, "
            f"EPC: {LEVELS['EPceeb12e_efus'][row['EPceeb12e_efus']]}, "
            f"Cooling: {LEVELS['AnyCooling'][row['AnyCooling']]}"
        )

        # Wrap title text to subplot width
        wrapped_text = textwrap.fill(char_text, width=45)

        ax.plot(
            df_plot["hour"],
            df_plot["T_out"],
            marker="x",
            label=r"$T_{out}$"
        )

        ax.plot(
            df_plot["hour"],
            df_plot["T_out_c"],
            marker="x",
            label=r"$T_{out,\ centered}$"
        )

        # ax.plot(
        #     df_plot["hour"],
        #     (
        #         df_plot
        #         .set_index("hour")["T_out"]
        #         .resample("D")
        #         .max()
        #         .rolling(2)
        #         .mean()
        #         .reindex(df_plot["hour"], method="ffill")
        #         .values
        #     ),
        #     linewidth=2,
        #     label="2-day running mean max $T_{out}$"
        # )

        ax.plot(
            df_plot["hour"],
            df_plot["T_in"],
            marker="x",
            label=r"$T_{in}$"
        )

        ax.fill_between(
            df_plot["hour"],
            26,
            35,
            color='black',
            alpha=0.2
        )

        # Format x-axis as MM-DD only
        ax.xaxis.set_major_formatter(
            mdates.DateFormatter("%m-%d")
        )

        ax.set_xlabel("Date")
        ax.set_ylabel(r"Temperature ($^\circ$C)")

        ax.set_title(
            f"Dwelling {random_dwelling} ({room_name})\n"
            f"{wrapped_text}",
            fontsize=8,
            pad=12,
            loc="center"
        )
        ax.legend(fontsize=8)



    # Remove unused axes
    for ax in axes[n:]:
        fig.delaxes(ax)

    # Extra spacing between subplots
    fig.subplots_adjust(
        hspace=0.5,
        wspace=0.1
    )

    

    plt.savefig(
        f"plots/efus2017/london_bedroom_4month/description/"
        f"T_out_multiple_{room_name}.svg",
        bbox_inches="tight",
        dpi=300
    )

    plt.show()

# Sample plots
# plot_temps(["1784", "310"])


In [ ]:
import matplotlib.dates as mdates
import textwrap

def plot_temps(dwelling_ids):

    # Ensure iterable
    if isinstance(dwelling_ids, str):
        dwelling_ids = [dwelling_ids]

    n = len(dwelling_ids)
    ncols = 2
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(12, 4.8 * nrows),
        squeeze=False
    )

    axes = axes.flatten()

    for ax, random_dwelling in zip(axes, dwelling_ids):

        # Subset data for that dwelling
        df_plot = (
            df_m[df_m["dwelling"] == str(random_dwelling)]
            .sort_values("hour")
        )

        # Building characteristics
        row = df_plot.iloc[0]

        char_text = (
            f"{LEVELS['dwtype_efus'][row['dwtype_efus']]}, "
            f"{LEVELS['dwage_efus'][row['dwage_efus']]}, "
            f"{LEVELS['WallType2x_efus'][row['WallType2x_efus']]}, "
            f"Insulated walls: "
            f"{LEVELS['InsulatedWalls_efus'][row['InsulatedWalls_efus']]}, "
            f"Double glazing: "
            f"{LEVELS['FullyDblGlz_efus'][row['FullyDblGlz_efus']]}, "
            f"Floor: {LEVELS['floor6x_efus'][row['floor6x_efus']]}, "
            f"EPC: {LEVELS['EPceeb12e_efus'][row['EPceeb12e_efus']]}, "
            f"Cooling: {LEVELS['AnyCooling'][row['AnyCooling']]}"
        )

        # Wrap title text to subplot width
        wrapped_text = textwrap.fill(char_text, width=45)

        ax.plot(
            df_plot["hour"],
            df_plot["T_out"],
            marker="x",
            label=r"$T_{out}$"
        )

        # ax.plot(
        #     df_plot["hour"],
        #     df_plot["T_out_c"],
        #     marker="x",
        #     label=r"$T_{out,\ centered}$"
        # )

        ax.plot(
            df_plot["hour"],
            (
                df_plot
                .set_index("hour")["T_out"]
                .resample("D")
                .max()
                .rolling(2)
                .mean()
                .reindex(df_plot["hour"], method="ffill")
                .values
            ),
            linewidth=2,
            label="2-day running mean max $T_{out}$"
        )

        ax.plot(
            df_plot["hour"],
            df_plot["T_in"],
            marker="x",
            label=r"$T_{in}$"
        )

        ax.fill_between(
            df_plot["hour"],
            26,
            35,
            color='black',
            alpha=0.2
        )

        # Format x-axis as MM-DD only
        ax.xaxis.set_major_formatter(
            mdates.DateFormatter("%m-%d")
        )

        ax.set_xlabel("Date")
        ax.set_ylabel(r"Temperature ($^\circ$C)")

        ax.set_title(
            f"Dwelling {random_dwelling} ({room_name})\n"
            f"{wrapped_text}",
            fontsize=8,
            pad=12,
            loc="center"
        )
        ax.legend(fontsize=8)
        # ax.set_ylim([5,30])
        # ax.set_xlim([209,220])

        # ax.set_xlim(
        #     pd.Timestamp("2018-08-21"),
        #     pd.Timestamp("2018-08-30")
        #     )
        
        ax.set_ylim([0,35])


    # Remove unused axes
    for ax in axes[n:]:
        fig.delaxes(ax)

    # Extra spacing between subplots
    fig.subplots_adjust(
        hspace=0.5,
        wspace=0.1
    )

    

    plt.savefig(
        f"plots/efus2017/london_bedroom_4month/description/"
        f"T_out_multiple_{room_name}_zoom.svg",
        bbox_inches="tight",
        dpi=300
    )

    plt.show()

# plot_temps(["2602"])

In [ ]:
# EPC T_{in} distribution violin plot 

import matplotlib.colors as mcolors

# EPC labels
EPC_AXIS_LABELS = {
    1: "C+",
    2: "D",
    3: "E",
    4: "F/G",
}

display_bands = [4, 3, 2, 1]

# Prepare EPC dataset
df_epc = df_m[["CaseID", "T_in", "EPceeb12e_efus"]].copy()

hourly_data = [
    df_epc[df_epc["EPceeb12e_efus"] == b]["T_in"].values
    for b in display_bands
]

xlabels = [EPC_AXIS_LABELS[b] for b in display_bands]

# Plot
fig, ax = plt.subplots(figsize=(3.4, 2.5))

parts = ax.violinplot(
    hourly_data,
    positions=range(1, 5),
    showmeans=False,
    showmedians=False,
    showextrema=True
)

# Style violins
dark_colors = []
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

for i, pc in enumerate(parts["bodies"]):

    color = colors[i % len(colors)]
    dark_color = tuple(np.array(mcolors.to_rgb(color)) * 0.45)

    dark_colors.append(dark_color)

    pc.set_facecolor(color)
    pc.set_edgecolor(color)
    pc.set_alpha(0.45)

# Keep only central spine
parts["cmins"].set_visible(False)
parts["cmaxes"].set_visible(False)
parts["cbars"].set_linewidth(1.2)
parts["cbars"].set_color(dark_colors)

# Add IQR box, median point, and labels
for i, data in enumerate(hourly_data, start=1):

    color = colors[i - 1]
    dark_color = dark_colors[i - 1]

    q1, med, q3 = np.percentile(data, [25, 50, 75])

    ax.add_patch(
        plt.Rectangle(
            (i - 0.06, q1),
            0.12,
            q3 - q1,
            facecolor=color,
            edgecolor=dark_color,
            linewidth=1.2,
            zorder=4
        )
    )

    # ax.scatter(
    #     i,
    #     med,
    #     color="white",
    #     edgecolor=dark_color,
    #     linewidth=0.6,
    #     s=26,
    #     zorder=5
    # )

    ax.plot(
        [i - 0.049, i + 0.049],
        [med, med],
        color="white",
        linewidth=1.2,
        solid_capstyle="butt",
        zorder=5
    )

    # Median label (aligned with median dot)
    ax.text(
        i + 0.23,
        med,
        f"{med:.1f}°C",
        fontsize=5,
        va="center",
        ha="center",
        color=dark_color
    )

    # IQR label
    y_offset = np.max(data) + 0.3

    ax.text(
        i,
        y_offset,
        f"IQR\n[{q1:.1f}, {q3:.1f}]",
        fontsize=5,
        va="bottom",
        ha="center",
        color=dark_color
    )

# Formatting
ymin = min(np.min(data) for data in hourly_data) - 1
ymax = max(np.max(data) for data in hourly_data) + 4
ax.set_ylim(ymin, ymax)
ax.set_xticks(range(1, 5))
ax.set_xticklabels(xlabels)
ax.tick_params(axis='x', which='minor', bottom=False, top=False)
ax.set_xlabel("EPC score band")
ax.set_ylabel(r"Indoor temperature, $T_{in}$ ($^\circ$C)")

ax.set_title(
    "Indoor"f" {room_name} ""temperature distributions by EPC score band\n"
    "EFUS 2017 London subset, May-September"
)

ax.grid(axis="y")

plt.tight_layout()
plt.savefig(f"plots/efus2017/london_bedroom_4month/description/epc_tin_distribution_{room_name}.svg", bbox_inches="tight")
plt.show()

In [ ]:
# AnyCooling T_{in} distribution violin plot

import matplotlib.colors as mcolors

# Cooling labels
COOLING_AXIS_LABELS = {
    0: "No Cooling",
    1: "Any Cooling"
}

display_groups = [0, 1]

# Prepare dataset
df_cooling = df_m[["CaseID", "T_in", "AnyCooling"]].copy()

hourly_data = [
    df_cooling[df_cooling["AnyCooling"] == g]["T_in"].values
    for g in display_groups
]

xlabels = [
    COOLING_AXIS_LABELS[g]
    for g in display_groups
]

# Plot
fig, ax = plt.subplots(figsize=(2.6, 2.5))

parts = ax.violinplot(
    hourly_data,
    positions=range(1, len(display_groups) + 1),
    showmeans=False,
    showmedians=False,
    showextrema=True
)

# Style violins
dark_colors = []
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

for i, pc in enumerate(parts["bodies"]):

    color = colors[i % len(colors)]

    dark_color = tuple(
        np.array(mcolors.to_rgb(color)) * 0.45
    )

    dark_colors.append(dark_color)

    pc.set_facecolor(color)
    pc.set_edgecolor(color)
    pc.set_alpha(0.45)

# Keep only central spine
parts["cmins"].set_visible(False)
parts["cmaxes"].set_visible(False)

parts["cbars"].set_linewidth(1.2)
parts["cbars"].set_color(dark_colors)

# Add IQR box, median line, and labels
for i, data in enumerate(hourly_data, start=1):

    color = colors[i - 1]
    dark_color = dark_colors[i - 1]

    q1, med, q3 = np.percentile(
        data,
        [25, 50, 75]
    )

    ax.add_patch(
        plt.Rectangle(
            (i - 0.06, q1),
            0.12,
            q3 - q1,
            facecolor=color,
            edgecolor=dark_color,
            linewidth=1.2,
            zorder=4
        )
    )

    ax.plot(
        [i - 0.053, i + 0.053],
        [med, med],
        color="white",
        linewidth=1.2,
        solid_capstyle="butt",
        zorder=5
    )

    # Median label
    ax.text(
        i + 0.23,
        med,
        f"{med:.1f}°C",
        fontsize=5,
        va="center",
        ha="center",
        color=dark_color
    )

    # IQR label
    y_offset = np.max(data) + 0.3

    ax.text(
        i,
        y_offset,
        f"IQR\n[{q1:.1f}, {q3:.1f}]",
        fontsize=5,
        va="bottom",
        ha="center",
        color=dark_color
    )

# Formatting
ymin = min(np.min(data) for data in hourly_data) - 1
ymax = max(np.max(data) for data in hourly_data) + 4

ax.set_ylim(ymin, ymax)

ax.set_xticks(
    range(1, len(display_groups) + 1)
)

ax.set_xticklabels(xlabels)

ax.tick_params(
    axis='x',
    which='minor',
    bottom=False,
    top=False
)

ax.set_xlabel("Presense of active cooling measures")

ax.set_ylabel(
    r"Indoor temperature, $T_{in}$ ($^\circ$C)"
)

ax.set_title(
    f"Indoor {room_name} temperature distributions by cooling presence\n"
    "EFUS 2017 London subset, May-September"
)

ax.grid(axis="y")

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/london_bedroom_4month/description/"
    f"cooling_tin_distribution_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# EPC T_{out_c} distribution violin plot 

import matplotlib.colors as mcolors

# EPC labels
EPC_AXIS_LABELS = {
    1: "C+",
    2: "D",
    3: "E",
    4: "F/G",
}

display_bands = [4, 3, 2, 1]

# Prepare EPC dataset
df_epc = df_m[["CaseID", "T_out_c", "EPceeb12e_efus"]].copy()

hourly_data = [
    df_epc[df_epc["EPceeb12e_efus"] == b]["T_out_c"].values
    for b in display_bands
]

xlabels = [EPC_AXIS_LABELS[b] for b in display_bands]


# Plot
fig, ax = plt.subplots(figsize=(3.4, 2.5))

parts = ax.violinplot(
    hourly_data,
    positions=range(1, 5),
    showmeans=False,
    showmedians=False,
    showextrema=True
)

# Style violins
dark_colors = []
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

for i, pc in enumerate(parts["bodies"]):

    color = colors[i % len(colors)]
    dark_color = tuple(np.array(mcolors.to_rgb(color)) * 0.45)

    dark_colors.append(dark_color)

    pc.set_facecolor(color)
    pc.set_edgecolor(color)
    pc.set_alpha(0.45)

# Keep only central spine
parts["cmins"].set_visible(False)
parts["cmaxes"].set_visible(False)
parts["cbars"].set_linewidth(1.2)
parts["cbars"].set_color(dark_colors)

# Add IQR box, median point, and labels
for i, data in enumerate(hourly_data, start=1):

    color = colors[i - 1]
    dark_color = dark_colors[i - 1]

    q1, med, q3 = np.percentile(data, [25, 50, 75])

    ax.add_patch(
        plt.Rectangle(
            (i - 0.06, q1),
            0.12,
            q3 - q1,
            facecolor=color,
            edgecolor=dark_color,
            linewidth=1.2,
            zorder=4
        )
    )

    # ax.scatter(
    #     i,
    #     med,
    #     color="white",
    #     edgecolor=dark_color,
    #     linewidth=0.6,
    #     s=26,
    #     zorder=5
    # )

    ax.plot(
        [i - 0.049, i + 0.049],
        [med, med],
        color="white",
        linewidth=1.2,
        solid_capstyle="butt",
        zorder=5
    )

    # Median label (aligned with median dot)
    ax.text(
        i + 0.23,
        med,
        f"{med:.1f}°C",
        fontsize=5,
        va="center",
        ha="center",
        color=dark_color
    )

    # IQR label
    y_offset = np.max(data) + 0.3

    ax.text(
        i,
        y_offset,
        f"IQR\n[{q1:.1f}, {q3:.1f}]",
        fontsize=5,
        va="bottom",
        ha="center",
        color=dark_color
    )

# Formatting
ymin = min(np.min(data) for data in hourly_data) - 1
ymax = max(np.max(data) for data in hourly_data) + 5
ax.set_ylim(ymin, ymax)
ax.set_xticks(range(1, 5))
ax.set_xticklabels(xlabels)
ax.tick_params(axis='x', which='minor', bottom=False, top=False)
ax.set_xlabel("EPC score band")
ax.set_ylabel(r"Centered outdoor temperature, $T_{out,c}$ ($^\circ$C)")

ax.set_title(
    "Centered outdoor"f" {room_name} ""temperature distributions by EPC score band\n"
    "EFUS 2017 London subset, May-September"
)

ax.grid(axis="y")

plt.tight_layout()
plt.savefig(f"plots/efus2017/london_bedroom_4month/description/epc_toutc_distribution_{room_name}.svg", bbox_inches="tight")
plt.show()

In [ ]:
# EPC daily maximum indoor temperature violin plot

import matplotlib.colors as mcolors

# EPC labels
EPC_AXIS_LABELS = {
    1: "C+",
    2: "D",
    3: "E",
    4: "F/G",
}

display_bands = [4, 3, 2, 1]

# Daily max indoor temperature
df_epc = df_m[[
    "CaseID",
    "hour",
    "T_in",
    "EPceeb12e_efus"
]].copy()

df_epc["date"] = df_epc["hour"].dt.floor("D")

daily_max = (
    df_epc
    .groupby(["CaseID", "date"])
    .agg(
        daily_max_T_in=("T_in", "max"),
        epc=("EPceeb12e_efus", "first")
    )
    .reset_index()
)

# EPC grouped data
hourly_data = [
    daily_max[daily_max["epc"] == b]["daily_max_T_in"].values
    for b in display_bands
]

xlabels = [EPC_AXIS_LABELS[b] for b in display_bands]

# Plot
fig, ax = plt.subplots(figsize=(3.4, 2.5))

parts = ax.violinplot(
    hourly_data,
    positions=range(1, 5),
    showmeans=False,
    showmedians=False,
    showextrema=True
)

# Style violins
dark_colors = []
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

for i, pc in enumerate(parts["bodies"]):

    color = colors[i % len(colors)]
    dark_color = tuple(np.array(mcolors.to_rgb(color)) * 0.45)

    dark_colors.append(dark_color)

    pc.set_facecolor(color)
    pc.set_edgecolor(color)
    pc.set_alpha(0.45)

# Keep only central spine
parts["cmins"].set_visible(False)
parts["cmaxes"].set_visible(False)
parts["cbars"].set_linewidth(1.2)
parts["cbars"].set_color(dark_colors)

# Add IQR box, median line, and labels
for i, data in enumerate(hourly_data, start=1):

    color = colors[i - 1]
    dark_color = dark_colors[i - 1]

    q1, med, q3 = np.percentile(data, [25, 50, 75])

    ax.add_patch(
        plt.Rectangle(
            (i - 0.06, q1),
            0.12,
            q3 - q1,
            facecolor=color,
            edgecolor=dark_color,
            linewidth=1.2,
            zorder=4
        )
    )

    # Median line
    ax.plot(
        [i - 0.049, i + 0.049],
        [med, med],
        color="white",
        linewidth=1.2,
        solid_capstyle="butt",
        zorder=5
    )

    # Median label
    ax.text(
        i + 0.23,
        med,
        f"{med:.1f}°C",
        fontsize=5,
        va="center",
        ha="center",
        color=dark_color
    )

    # IQR label
    y_offset = np.max(data) + 0.3

    ax.text(
        i,
        y_offset,
        f"IQR\n[{q1:.1f}, {q3:.1f}]",
        fontsize=5,
        va="bottom",
        ha="center",
        color=dark_color
    )

# Formatting
ymin = min(np.min(data) for data in hourly_data) - 1
ymax = max(np.max(data) for data in hourly_data) + 3

ax.set_ylim(ymin, ymax)
ax.set_xticks(range(1, 5))
ax.set_xticklabels(xlabels)
ax.set_yticks(np.arange(np.floor(ymin) + 1, np.floor(ymax), 3))

ax.set_xlabel("EPC score band")

ax.set_ylabel(
    "Daily maximum indoor temperature\n"
    r"$T_{in,max}$ ($^\circ$C)"
)

ax.set_title(
    f"Daily maximum indoor {room_name} temperature distributions by EPC score band\n"
    "EFUS 2017 London subset, May-September"
)

ax.grid(axis="y")

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/london_bedroom_4month/description/"
    f"epc_daily_max_tin_distribution_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()

# Statistical comparison of daily maximum indoor temperatures
# Compare C+ against all other EPC bands

from scipy.stats import mannwhitneyu, ks_2samp, levene

print("\nStatistical comparison against EPC C+\n")

# Reference group: C+
ref_data = daily_max[
    daily_max["epc"] == 1
]["daily_max_T_in"].values

for b in [2, 3, 4]:

    comp_data = daily_max[
        daily_max["epc"] == b
    ]["daily_max_T_in"].values

    band_name = EPC_AXIS_LABELS[b].split("\n")[0]

    # 1. Mann–Whitney U test (difference in distributions / medians)
    mw_stat, mw_p = mannwhitneyu(
        ref_data,
        comp_data,
        alternative="greater"
    )

    # 2. Kolmogorov–Smirnov test (cumfreq differnece)
    ks_stat, ks_p = ks_2samp(
        ref_data,
        comp_data,
        alternative="greater"
    )

    # 3. Levene test (difference in variability / spread)
    lev_stat, lev_p = levene(
        ref_data,
        comp_data,
        center="median"
    )

    print(f"\nC+ vs {band_name}")
    print("-" * 40)

    print(
        f"Mann–Whitney U : "
        f"U = {mw_stat:.1f}, p = {mw_p:.4g}"
    )

    print(
        f"Kolmogorov–Smirnov : "
        f"D = {ks_stat:.3f}, p = {ks_p:.4g}"
    )

    print(
        f"Levene (IQR/spread) : "
        f"W = {lev_stat:.3f}, p = {lev_p:.4g}"
    )

In [ ]:
# 2DMMT by EPC

import matplotlib.colors as mcolors

# EPC labels
EPC_AXIS_LABELS = {
    1: "C+",
    2: "D",
    3: "E",
    4: "F/G",
}

display_bands = [4, 3, 2, 1]

# 2DMMT calculation
df_2dmmt = df_m[[
    "CaseID",
    "hour",
    "T_out",
    "EPceeb12e_efus"
]].copy()

# Daily maximum outdoor temperature
df_2dmmt["date"] = df_2dmmt["hour"].dt.floor("D")

daily_max = (
    df_2dmmt
    .groupby(["CaseID", "date"])
    .agg(
        daily_max_T_out=("T_out", "max"),
        epc=("EPceeb12e_efus", "first")
    )
    .reset_index()
)

# 2-day running mean of daily maxima
daily_max = daily_max.sort_values(["CaseID", "date"])

daily_max["T_2DMMT"] = (
    daily_max
    .groupby("CaseID")["daily_max_T_out"]
    .transform(lambda x: x.rolling(2, min_periods=1).mean())
)

# EPC grouped data
hourly_data = [
    daily_max[daily_max["epc"] == b]["T_2DMMT"].values
    for b in display_bands
]

xlabels = [EPC_AXIS_LABELS[b] for b in display_bands]

# Plot
fig, ax = plt.subplots(figsize=(3.4, 2.5))

parts = ax.violinplot(
    hourly_data,
    positions=range(1, 5),
    showmeans=False,
    showmedians=False,
    showextrema=True
)

# Style violins
dark_colors = []
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

for i, pc in enumerate(parts["bodies"]):

    color = colors[i % len(colors)]
    dark_color = tuple(np.array(mcolors.to_rgb(color)) * 0.45)

    dark_colors.append(dark_color)

    pc.set_facecolor(color)
    pc.set_edgecolor(color)
    pc.set_alpha(0.45)

# Keep only central spine
parts["cmins"].set_visible(False)
parts["cmaxes"].set_visible(False)
parts["cbars"].set_linewidth(1.2)
parts["cbars"].set_color(dark_colors)

# Add IQR box, median line, and labels
for i, data in enumerate(hourly_data, start=1):

    color = colors[i - 1]
    dark_color = dark_colors[i - 1]

    q1, med, q3 = np.percentile(data, [25, 50, 75])

    ax.add_patch(
        plt.Rectangle(
            (i - 0.06, q1),
            0.12,
            q3 - q1,
            facecolor=color,
            edgecolor=dark_color,
            linewidth=1.2,
            zorder=4
        )
    )

    ax.plot(
        [i - 0.049, i + 0.049],
        [med, med],
        color="white",
        linewidth=1.2,
        solid_capstyle="butt",
        zorder=5
    )

    # Median label
    ax.text(
        i + 0.23,
        med,
        f"{med:.1f}°C",
        fontsize=5,
        va="center",
        ha="center",
        color=dark_color
    )

    # IQR label
    y_offset = np.max(data) + 0.3

    ax.text(
        i,
        y_offset,
        f"IQR\n[{q1:.1f}, {q3:.1f}]",
        fontsize=5,
        va="bottom",
        ha="center",
        color=dark_color
    )

# Formatting
ymin = min(np.min(data) for data in hourly_data) - 1
ymax = max(np.max(data) for data in hourly_data) + 3

ax.set_ylim(ymin, ymax)
ax.set_xticks(range(1, 5))
ax.set_xticklabels(xlabels)

ax.tick_params(axis="x", which="minor", bottom=False, top=False)

ax.set_xlabel("EPC score band")
ax.set_ylabel(r"Outdoor Two-day Mean Max Temperature (2DMMT) ($^\circ$C)")

ax.set_title(
    f"Outdoor 2DMMT distributions by EPC score band\n"
    f"EFUS 2017 London subset, May-September"
)

ax.grid(axis="y")

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/london_bedroom_4month/description/epc_2dmmt_distribution_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# 2DMMT by AnyCooling

import matplotlib.colors as mcolors

# Cooling labels
COOLING_AXIS_LABELS = {
    0: "No Cooling",
    1: "Any Cooling"
}

display_groups = [0, 1]

# 2DMMT calculation
df_2dmmt = df_m[[
    "CaseID",
    "hour",
    "T_out",
    "AnyCooling"
]].copy()

# Daily maximum outdoor temperature
df_2dmmt["date"] = df_2dmmt["hour"].dt.floor("D")

daily_max = (
    df_2dmmt
    .groupby(["CaseID", "date"])
    .agg(
        daily_max_T_out=("T_out", "max"),
        cooling=("AnyCooling", "first")
    )
    .reset_index()
)

# 2-day running mean of daily maxima
daily_max = daily_max.sort_values(
    ["CaseID", "date"]
)

daily_max["T_2DMMT"] = (
    daily_max
    .groupby("CaseID")["daily_max_T_out"]
    .transform(
        lambda x: x.rolling(
            2,
            min_periods=1
        ).mean()
    )
)

# Cooling grouped data
hourly_data = [
    daily_max[
        daily_max["cooling"] == g
    ]["T_2DMMT"].values
    for g in display_groups
]

xlabels = [
    COOLING_AXIS_LABELS[g]
    for g in display_groups
]

# Plot
fig, ax = plt.subplots(
    figsize=(2.6, 2.5)
)

parts = ax.violinplot(
    hourly_data,
    positions=range(
        1,
        len(display_groups) + 1
    ),
    showmeans=False,
    showmedians=False,
    showextrema=True
)

# Style violins
dark_colors = []

colors = plt.rcParams[
    "axes.prop_cycle"
].by_key()["color"]

for i, pc in enumerate(parts["bodies"]):

    color = colors[i % len(colors)]

    dark_color = tuple(
        np.array(
            mcolors.to_rgb(color)
        ) * 0.45
    )

    dark_colors.append(dark_color)

    pc.set_facecolor(color)
    pc.set_edgecolor(color)
    pc.set_alpha(0.45)

# Keep only central spine
parts["cmins"].set_visible(False)
parts["cmaxes"].set_visible(False)

parts["cbars"].set_linewidth(1.2)
parts["cbars"].set_color(dark_colors)

# Add IQR box, median line, and labels
for i, data in enumerate(hourly_data, start=1):

    color = colors[i - 1]
    dark_color = dark_colors[i - 1]

    q1, med, q3 = np.percentile(
        data,
        [25, 50, 75]
    )

    ax.add_patch(
        plt.Rectangle(
            (i - 0.06, q1),
            0.12,
            q3 - q1,
            facecolor=color,
            edgecolor=dark_color,
            linewidth=1.2,
            zorder=4
        )
    )

    ax.plot(
        [i - 0.049, i + 0.049],
        [med, med],
        color="white",
        linewidth=1.2,
        solid_capstyle="butt",
        zorder=5
    )

    # Median label
    ax.text(
        i + 0.23,
        med,
        f"{med:.1f}°C",
        fontsize=5,
        va="center",
        ha="center",
        color=dark_color
    )

    # IQR label
    y_offset = np.max(data) + 0.3

    ax.text(
        i,
        y_offset,
        f"IQR\n[{q1:.1f}, {q3:.1f}]",
        fontsize=5,
        va="bottom",
        ha="center",
        color=dark_color
    )

# Formatting
ymin = min(np.min(data) for data in hourly_data) - 1
ymax = max(np.max(data) for data in hourly_data) + 3

ax.set_ylim(ymin, ymax)

ax.set_xticks(
    range(
        1,
        len(display_groups) + 1
    )
)

ax.set_xticklabels(xlabels)

ax.tick_params(
    axis="x",
    which="minor",
    bottom=False,
    top=False
)

ax.set_xlabel("Presence of active cooling measures")

ax.set_ylabel(
    r"Outdoor Two-day Mean Max Temperature (2DMMT) ($^\circ$C)"
)

ax.set_title(
    f"Outdoor 2DMMT distributions by cooling presence\n"
    f"EFUS 2017 London subset, May-September"
)

ax.grid(axis="y")

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/london_bedroom_4month/description/"
    f"cooling_2dmmt_distribution_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# 2DMMnT by EPC

import matplotlib.colors as mcolors

# EPC labels
EPC_AXIS_LABELS = {
    1: "C+",
    2: "D",
    3: "E",
    4: "F/G",
}

display_bands = [4, 3, 2, 1]

# Calculate 2-day mean minimum indoor temperature (2DMMnT) per dwelling
df_2dmmnt = df_m[[
    "CaseID",
    "hour",
    "T_in",
    "EPceeb12e_efus"
]].copy()

df_2dmmnt["date"] = df_2dmmnt["hour"].dt.floor("D")

daily_min = (
    df_2dmmnt
    .groupby(["CaseID", "date"])
    .agg(
        daily_min_T_in=("T_in", "min"),
        epc=("EPceeb12e_efus", "first")
    )
    .reset_index()
)

# 2-day running mean of daily indoor minima
daily_min = daily_min.sort_values(["CaseID", "date"])

daily_min["T_2DMMnT"] = (
    daily_min
    .groupby("CaseID")["daily_min_T_in"]
    .transform(lambda x: x.rolling(2, min_periods=1).mean())
)

# EPC grouped data
hourly_data = [
    daily_min[daily_min["epc"] == b]["T_2DMMnT"].values
    for b in display_bands
]

xlabels = [EPC_AXIS_LABELS[b] for b in display_bands]

# Plot
fig, ax = plt.subplots(figsize=(3.4, 2.5))

parts = ax.violinplot(
    hourly_data,
    positions=range(1, 5),
    showmeans=False,
    showmedians=False,
    showextrema=True
)

# Style violins
dark_colors = []
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

for i, pc in enumerate(parts["bodies"]):

    color = colors[i % len(colors)]
    dark_color = tuple(np.array(mcolors.to_rgb(color)) * 0.45)

    dark_colors.append(dark_color)

    pc.set_facecolor(color)
    pc.set_edgecolor(color)
    pc.set_alpha(0.45)

# Keep only central spine
parts["cmins"].set_visible(False)
parts["cmaxes"].set_visible(False)
parts["cbars"].set_linewidth(1.2)
parts["cbars"].set_color(dark_colors)

# Add IQR box, median line, and labels
for i, data in enumerate(hourly_data, start=1):

    color = colors[i - 1]
    dark_color = dark_colors[i - 1]

    q1, med, q3 = np.percentile(data, [25, 50, 75])

    ax.add_patch(
        plt.Rectangle(
            (i - 0.06, q1),
            0.12,
            q3 - q1,
            facecolor=color,
            edgecolor=dark_color,
            linewidth=1.2,
            zorder=4
        )
    )

    ax.plot(
        [i - 0.049, i + 0.049],
        [med, med],
        color="white",
        linewidth=1.2,
        solid_capstyle="butt",
        zorder=5
    )

    # Median label
    ax.text(
        i + 0.23,
        med,
        f"{med:.1f}°C",
        fontsize=5,
        va="center",
        ha="center",
        color=dark_color
    )

    # IQR label
    y_offset = np.max(data) + 0.3

    ax.text(
        i,
        y_offset,
        f"IQR\n[{q1:.1f}, {q3:.1f}]",
        fontsize=5,
        va="bottom",
        ha="center",
        color=dark_color
    )

# Formatting
ymin = min(np.min(data) for data in hourly_data) - 1
ymax = max(np.max(data) for data in hourly_data) + 3

ax.set_ylim(ymin, ymax)
ax.set_xticks(range(1, 5))
ax.set_xticklabels(xlabels)

ax.tick_params(axis="x", which="minor", bottom=False, top=False)

ax.set_xlabel("EPC score band")
ax.set_ylabel("Two-day Mean Min Temperature\n(2DMMnT) ($^\\circ$C)")

ax.set_title(
    f"Indoor {room_name} 2DMMnT distributions by EPC score band\n"
    f"EFUS 2017 London subset, May-September"
)

ax.grid(axis="y")

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/london_bedroom_4month/description/epc_in_2dmmnt_distribution_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# 2DMMT + 2DMMnT by EPC

import matplotlib.colors as mcolors

# EPC labels
EPC_AXIS_LABELS = {
    1: "C+",
    2: "D",
    3: "E",
    4: "F/G",
}

display_bands = [4, 3, 2, 1]

# Prepare dataset
# T_2DMMT: 2-day mean of daily outdoor max (climate forcing indicator)
# T_2DMMnT: 2-day mean of daily indoor min (health-relevant overnight low)
df_temp = df_m[[
    "CaseID",
    "hour",
    "T_out",
    "T_in",
    "EPceeb12e_efus"
]].copy()

df_temp["date"] = df_temp["hour"].dt.floor("D")

daily_stats = (
    df_temp
    .groupby(["CaseID", "date"])
    .agg(
        daily_max_T_out=("T_out", "max"),
        daily_min_T_in=("T_in",  "min"),
        epc=("EPceeb12e_efus", "first")
    )
    .reset_index()
    .sort_values(["CaseID", "date"])
)

# Rolling metrics
daily_stats["T_2DMMT"] = (
    daily_stats
    .groupby("CaseID")["daily_max_T_out"]
    .transform(lambda x: x.rolling(2, min_periods=1).mean())
)

daily_stats["T_2DMMnT"] = (
    daily_stats
    .groupby("CaseID")["daily_min_T_in"]
    .transform(lambda x: x.rolling(2, min_periods=1).mean())
)

# Shared style
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

# Plotting helper
def epc_violin_plot(ax, metric, ylabel, title):

    hourly_data = [
        daily_stats[daily_stats["epc"] == b][metric].values
        for b in display_bands
    ]

    xlabels = [EPC_AXIS_LABELS[b] for b in display_bands]

    parts = ax.violinplot(
        hourly_data,
        positions=range(1, 5),
        showmeans=False,
        showmedians=False,
        showextrema=True
    )

    dark_colors = []

    # Style violins
    for i, pc in enumerate(parts["bodies"]):

        color = colors[i % len(colors)]
        dark_color = tuple(np.array(mcolors.to_rgb(color)) * 0.45)

        dark_colors.append(dark_color)

        pc.set_facecolor(color)
        pc.set_edgecolor(color)
        pc.set_alpha(0.45)

    # Central spine only
    parts["cmins"].set_visible(False)
    parts["cmaxes"].set_visible(False)
    parts["cbars"].set_linewidth(1.2)
    parts["cbars"].set_color(dark_colors)

    # IQR + median
    for i, data in enumerate(hourly_data, start=1):

        color = colors[i - 1]
        dark_color = dark_colors[i - 1]

        q1, med, q3 = np.percentile(data, [25, 50, 75])

        ax.add_patch(
            plt.Rectangle(
                (i - 0.06, q1),
                0.12,
                q3 - q1,
                facecolor=color,
                edgecolor=dark_color,
                linewidth=1.2,
                zorder=4
            )
        )

        ax.plot(
            [i - 0.049, i + 0.049],
            [med, med],
            color="white",
            linewidth=1.2,
            solid_capstyle="butt",
            zorder=5
        )

        # Median label
        ax.text(
            i + 0.23,
            med,
            f"{med:.1f}°C",
            fontsize=5,
            va="center",
            ha="center",
            color=dark_color
        )

        # IQR label
        y_offset = np.max(data) + 0.3

        ax.text(
            i,
            y_offset,
            f"IQR\n[{q1:.1f}, {q3:.1f}]",
            fontsize=5,
            va="bottom",
            ha="center",
            color=dark_color
        )

    # Formatting
    ymin = min(np.min(data) for data in hourly_data) - 1
    ymax = max(np.max(data) for data in hourly_data) + 3

    ax.set_ylim(ymin, ymax)
    ax.set_xticks(range(1, 5))
    ax.set_xticklabels(xlabels)
    ax.tick_params(axis="x", which="minor", bottom=False, top=False)

    ax.set_xlabel("EPC score band")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y")

# 2DMMT
fig, ax = plt.subplots(figsize=(3.4, 2.5))

epc_violin_plot(
    ax=ax,
    metric="T_2DMMT",
    ylabel="Outdoor Two-day Mean Max Temperature\n(2DMMT) ($^\\circ$C)",
    title=(
        f"Outdoor 2DMMT distributions by EPC score band\n"
        f"EFUS 2017 London subset, May-September"
    )
)

plt.tight_layout()
plt.savefig(
    f"plots/efus2017/london_bedroom_4month/description/epc_2dmmt_distribution_{room_name}.svg",
    bbox_inches="tight"
)
plt.show()

# 2DMMnt
fig, ax = plt.subplots(figsize=(3.4, 2.5))

epc_violin_plot(
    ax=ax,
    metric="T_2DMMnT",
    ylabel="Indoor Two-day Mean Min Temperature\n(2DMMnT) ($^\\circ$C)",
    title=(
        f"Indoor {room_name} 2DMMnT distributions by EPC score band\n"
        f"EFUS 2017 London subset, May-September"
    )
)

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/london_bedroom_4month/description/epc_2dmmnt_distribution_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# EPC violin plots: T_in, T_out, T_out_c, 2DMMT, 2DMMnT

import matplotlib.colors as mcolors

# EPC labels
# EPC_AXIS_LABELS = {
#     1: "C+\n(SAP $>70$)",
#     2: "D\n(SAP 51–70)",
#     3: "E\n(SAP 30–50)",
#     4: "F/G\n(SAP $<30$)",
# }

EPC_AXIS_LABELS = {
    1: "C+",
    2: "D",
    3: "E",
    4: "F/G",
}

display_bands = [4, 3, 2, 1]
xlabels = [EPC_AXIS_LABELS[b] for b in display_bands]

# Shared plotting style
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

# Base building characteristics 
df_epc = df_m[[
    "CaseID",
    "hour",
    "T_in",
    "T_out",
    "T_out_c",
    "EPceeb12e_efus"
]].copy()

# 2DMMT and 2DMMnT computation
df_epc["date"] = df_epc["hour"].dt.floor("D")

daily_stats = (
    df_epc
    .groupby(["CaseID", "date"])
    .agg(
        daily_max_T_out=("T_out", "max"),
        daily_min_T_out=("T_out", "min"),
        epc=("EPceeb12e_efus", "first")
    )
    .reset_index()
    .sort_values(["CaseID", "date"])
)

daily_stats["T_2DMMT"] = (
    daily_stats
    .groupby("CaseID")["daily_max_T_out"]
    .transform(lambda x: x.rolling(2, min_periods=1).mean())
)

daily_stats["T_2DMMnT"] = (
    daily_stats
    .groupby("CaseID")["daily_min_T_out"]
    .transform(lambda x: x.rolling(2, min_periods=1).mean())
)

# Violin plot function
def epc_violin_plot(data_groups, ylabel, title, save_path):
    
    # LaTeX article size 
    fig, ax = plt.subplots(figsize=(3.4, 2.5))

    parts = ax.violinplot(
        data_groups,
        positions=range(1, 5),
        showmeans=False,
        showmedians=False,
        showextrema=True
    )

    dark_colors = []

    # Style violins
    for i, pc in enumerate(parts["bodies"]):

        color = colors[i % len(colors)]
        dark_color = tuple(np.array(mcolors.to_rgb(color)) * 0.45)

        dark_colors.append(dark_color)

        pc.set_facecolor(color)
        pc.set_edgecolor(color)
        pc.set_alpha(0.45)

    # Central spine only
    parts["cmins"].set_visible(False)
    parts["cmaxes"].set_visible(False)
    parts["cbars"].set_linewidth(1.2)
    parts["cbars"].set_color(dark_colors)

    # IQR box + labels
    for i, data in enumerate(data_groups, start=1):

        color = colors[i - 1]
        dark_color = dark_colors[i - 1]

        q1, med, q3 = np.percentile(data, [25, 50, 75])

        ax.add_patch(
            plt.Rectangle(
                (i - 0.06, q1),
                0.12,
                q3 - q1,
                facecolor=color,
                edgecolor=dark_color,
                linewidth=1.2,
                zorder=4
            )
        )

        # Median line
        ax.plot(
            [i - 0.044, i + 0.044],
            [med, med],
            color="white",
            linewidth=1.2,
            solid_capstyle="butt",
            zorder=5
        )

        # Median label
        ax.text(
            i + 0.23,
            med-0.02,
            f"{med:.1f}°C",
            fontsize=5,
            va="center",
            ha="center",
            color=dark_color
        )

        # # IQR label
        # x_offset = 0.4 if i < len(data_groups) else -0.4
        # y_offset = q1 - 0.5 if i < len(data_groups) else q3 + 0.5

        # ax.text(
        #     i + x_offset,
        #     y_offset,
        #     f"IQR\n[{q1:.1f}, {q3:.1f}] °C",
        #     fontsize=5,
        #     va="top",
        #     ha="center",
        #     color=dark_color
        # )


        # IQR label
        y_offset = np.max(data) + 1

        ax.text(
            i,
            y_offset,
            f"IQR\n[{q1:.1f}, {q3:.1f}]",
            fontsize=5,
            va="bottom",
            ha="center",
            color=dark_color
        )
    # Formatting
    ymin = min(np.min(data) for data in data_groups) - 1
    ymax = max(np.max(data) for data in data_groups) + 5

    ax.set_ylim(ymin, ymax)
    ax.set_xticks(range(1, 5))
    ax.set_xticklabels(xlabels)
    ax.tick_params(axis="x", which="minor", bottom=False, top=False)
    ax.set_xlabel("EPC score band")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y")

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight")
    plt.show()

# Internal T
hourly_data = [
    df_epc[df_epc["EPceeb12e_efus"] == b]["T_in"].values
    for b in display_bands
]

epc_violin_plot(
    data_groups=hourly_data,
    ylabel=r"Indoor temperature, $T_{in}$ ($^\circ$C)",
    title=(
        f"Indoor {room_name} temperature distributions by EPC score band\n"
        "EFUS 2017 London subset, May-September"
    ),
    save_path=f"plots/efus2017/london_bedroom_4month/description/epc_tin_distribution_{room_name}.svg"
)

# External T
hourly_data = [
    df_epc[df_epc["EPceeb12e_efus"] == b]["T_out"].values
    for b in display_bands
]

epc_violin_plot(
    data_groups=hourly_data,
    ylabel=r"Outdoor temperature, $T_{out}$ ($^\circ$C)",
    title=(
        f"Outdoor temperature distributions by EPC score band\n"
        "EFUS 2017 London subset, May-September"
    ),
    save_path=f"plots/efus2017/london_bedroom_4month/description/epc_tout_distribution_{room_name}.svg"
)

# External T (centered)
hourly_data = [
    df_epc[df_epc["EPceeb12e_efus"] == b]["T_out_c"].values
    for b in display_bands
]

epc_violin_plot(
    data_groups=hourly_data,
    ylabel=r"Centered outdoor temperature, $T_{out,c}$ ($^\circ$C)",
    title=(
        f"Centered outdoor temperature distributions by EPC score band\n"
        "EFUS 2017 London subset, May-September"
    ),
    save_path=f"plots/efus2017/london_bedroom_4month/description/epc_toutc_distribution_{room_name}.svg"
)

# 2DMMT
hourly_data = [
    daily_stats[daily_stats["epc"] == b]["T_2DMMT"].values
    for b in display_bands
]

epc_violin_plot(
    data_groups=hourly_data,
    ylabel="Outdoor Two-day Mean Max Temperature\n(2DMMT) ($^\\circ$C)",
    title=(
        f"2DMMT distributions by EPC score band\n"
        "EFUS 2017 London subset, May-September"
    ),
    save_path=f"plots/efus2017/london_bedroom_4month/description/epc_2dmmt_distribution_{room_name}.svg"
)

# Outdoor 2DMMnT
hourly_data = [
    daily_stats[daily_stats["epc"] == b]["T_2DMMnT"].values
    for b in display_bands
]

epc_violin_plot(
    data_groups=hourly_data,
    ylabel="Outdoor Two-day Mean Min Temperature\n(2DMMnT) ($^\\circ$C)",
    title=(
        f"Outdoor 2DMMnT distributions by EPC score band\n"
        "EFUS 2017 London subset, May-September"
    ),
    save_path=f"plots/efus2017/london_bedroom_4month/description/epc_out_2dmmnt_distribution_{room_name}.svg"
)

In [ ]:
""" Exceedance for the bedroom
Following the fixed-threshold criterion, bedroom overheating is >1% of occupied
night hours (22:00-06:59) above the threshold temperature.
This means 9 occupied hours/day for hours 22, 23, 0-6 inclusive.
Nights are attributed to calendar date (matching efus_overheating_datasets.py).
"""

# Occupied hours only
df_occ = df_m[
    df_m["hour"].dt.hour.isin(list(range(22, 24)) + list(range(0, 7)))
].copy()

# Add date + 2DMMT to occupied-hour dataframe for exceedance plotting

df_occ["date"] = df_occ["hour"].dt.floor("D")

df_occ = df_occ.drop(columns=["T_2DMMT"], errors="ignore")

df_occ = df_occ.merge(
    daily_stats[["CaseID", "date", "T_2DMMT"]],
    on=["CaseID", "date"],
    how="left"
)

# Thresholds to test
thresholds = [26, 27, 28]

# Exceedance frequency
results = []

for threshold in thresholds:

    exceed = (
        df_occ["T_in"] >= threshold
    ).astype(int)

    tmp = df_occ.copy()
    tmp["exceed"] = exceed

    # Frequency of exceedance per dwelling
    freq = (
        tmp.groupby("CaseID")
        .agg(
            exceed_hours=("exceed", "sum"),
            occupied_hours=("exceed", "count")
        )
        .reset_index()
    )

    freq["exceedance_pct"] = (
        100 * freq["exceed_hours"] / freq["occupied_hours"]
    )

    # Fixed-threshold overheating criterion (>1% of occupied night hours)
    freq["overheating"] = freq["exceedance_pct"] > 1

    # Summary statistics
    n_overheat = freq["overheating"].sum()
    n_total = len(freq)

    results.append({
        "threshold": threshold,
        "n_overheat": n_overheat,
        "n_total": n_total,
        "pct_overheat": 100 * n_overheat / n_total,
        "median_exceedance": freq["exceedance_pct"].median(),
        "iqr_low": freq["exceedance_pct"].quantile(0.25),
        "iqr_high": freq["exceedance_pct"].quantile(0.75)
    })

results_df = pd.DataFrame(results)

print(f"\nFixed-threshold {room_name} exceedance analysis\n")

for _, row in results_df.iterrows():

    print(
        f"{row['threshold']:.0f}°C : "
        f"{row['n_overheat']:.0f}/{row['n_total']:.0f} dwellings "
        f"({row['pct_overheat']:.1f}%) exceeded overheating criterion | "
        f"Median exceedance = {row['median_exceedance']:.1f}% "
        f"[IQR {row['iqr_low']:.1f}–{row['iqr_high']:.1f}%]"
    )

In [ ]:
# # fixed threshold exceedance vs 2DMMT by EPC band
# # Assumes df_occ, thresholds, and T_2DMMT already exist from previous cells
# import matplotlib.ticker as mtick

# fig, ax = plt.subplots(figsize=(3.4, 2.6))
# colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

# for epc in [1, 2, 3, 4]:

#     df_epc = df_occ[df_occ["EPceeb12e_efus"] == epc]
#     epc_color = colors[4 - epc]

#     for i, threshold in enumerate(thresholds):

#         tmp = df_epc.copy()

#         # Threshold exceedance
#         tmp["exceed"] = (
#             tmp["T_in"] >= threshold
#         ).astype(int)

#         # Dwelling-day exceedance
#         freq = (
#             tmp.groupby(["CaseID", "date", "T_2DMMT"])
#             .agg(
#                 exceed_hours=("exceed", "sum"),
#                 occupied_hours=("exceed", "count")
#             )
#             .reset_index()
#         )

#         freq["exceedance_pct"] = (
#             100 * freq["exceed_hours"] / freq["occupied_hours"]
#         )

#         freq["overheating"] = (
#             freq["exceedance_pct"] > 1
#         )

#         # Rounded 2DMMT bins
#         freq["T_2DMMT_bin"] = (
#             freq["T_2DMMT"].round()
#         )

#         prop = (
#             freq.groupby("T_2DMMT_bin")["overheating"]
#             .mean()
#             .reset_index()
#         )

#         ax.plot(
#             prop["T_2DMMT_bin"],
#             100 * prop["overheating"],
#             marker="o",
#             markersize=2.5,
#             linewidth=1.2,
#             linestyle=["-", "--", ":", "-."][i],
#             color=epc_color,
#             label=(
#                 f"{EPC_AXIS_LABELS[epc].split(chr(10))[0]} "
#                 f"| {threshold}°C"
#             )
#         )

# # Formatting
# ax.set_xlabel("Outdoor Two-day Mean Max Temperature (2DMMT) ($^\\circ$C)")
# ax.set_ylabel(
#     r"Dwellings exceeding fixed threshold (\%)"
# )
# ax.set_ylim(0, 100)
# ax.set_title(
#     f"Indoor {room_name} fixed threshold exceedance by EPC score band and 2DMMT by EPC score band and 2DMMT\n"
#         "EFUS 2017 London subset, May-September"
# )

# ax.yaxis.set_major_formatter(mtick.PercentFormatter())
# ax.grid(axis="y")
# ax.legend(
#     fontsize=4.5,
#     ncol=2,
#     frameon=True
# )

# plt.tight_layout()

# plt.savefig(
#     f"plots/efus2017/london_bedroom_4month/description/"
#     f"fixed_exceedance_vs_2dmmt_{room_name}.svg",
#     bbox_inches="tight"
# )

# plt.show()

In [ ]:
# fixed threshold exceedance vs 2DMMT
# Separate subplot for each temperature threshold

import matplotlib.ticker as mtick

fig, axes = plt.subplots(
    1, len(thresholds),
    figsize=(6.8, 2.5),
    sharex=True,
    sharey=True
)

axes = axes.flatten()

colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

for threshold, ax in zip(thresholds, axes):

    for epc in [1, 2, 3, 4]:

        df_epc = df_occ[df_occ["EPceeb12e_efus"] == epc]
        epc_color = colors[4 - epc]

        tmp = df_epc.copy()

        # Threshold exceedance
        tmp["exceed"] = (
            tmp["T_in"] >= threshold
        ).astype(int)

        # Dwelling-day exceedance
        freq = (
            tmp.groupby(["CaseID", "date", "T_2DMMT"])
            .agg(
                exceed_hours=("exceed", "sum"),
                occupied_hours=("exceed", "count")
            )
            .reset_index()
        )

        freq["exceedance_pct"] = (
            100 * freq["exceed_hours"] / freq["occupied_hours"]
        )

        freq["overheating"] = (
            freq["exceedance_pct"] > 1
        )

        # Rounded 2DMMT bins
        freq["T_2DMMT_bin"] = (
            freq["T_2DMMT"].round()
        )

        prop = (
            freq.groupby("T_2DMMT_bin")["overheating"]
            .mean()
            .reset_index()
        )

        ax.plot(
            prop["T_2DMMT_bin"],
            100 * prop["overheating"],
            marker="o",
            markersize=2.5,
            linewidth=1.4,
            color=epc_color,
            label=EPC_AXIS_LABELS[epc].split(chr(10))[0]
        )

    # Subplot formatting
    ax.set_title(
        f"{threshold}$^\\circ$C threshold",
        fontsize=8
    )

    ax.tick_params(axis="x", labelbottom=True)
    
    ax.set_ylim(0, 100)

    ax.yaxis.set_major_formatter(
        mtick.PercentFormatter()
    )

    ax.grid(axis="y")

# Shared labels
# fig.supxlabel(
#     "Outdoor Two-day Mean Max Temperature (2DMMT) ($^\\circ$C)"
# )
fig.supxlabel(
    "Outdoor Two-day Mean Max Temperature (2DMMT) ($^\\circ$C)")

fig.supylabel(
    r"Dwellings exceeding fixed threshold (\%)"
)

# Shared legend
handles, labels = axes[0].get_legend_handles_labels()

fig.suptitle(
    f"Indoor {room_name} fixed threshold exceedance by EPC score band and outdoor 2DMMT\n"
    "EFUS 2017 London subset, May-September",
    y=1.03
)

fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=4,
    frameon=True,
    fontsize=6,
    title="EPC band",
    bbox_to_anchor=(0.5, -0.15)
)

plt.tight_layout()
# fig.subplots_adjust(bottom=0.3)

plt.savefig(
    f"plots/efus2017/london_bedroom_4month/description/"
    f"fixed_exceedance_vs_2dmmt_threshold_subplots_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()

# Cumulative fixed threshold exceedance vs 2DMMT
# Separate subplot for each temperature threshold

import matplotlib.ticker as mtick

fig, axes = plt.subplots(
    1, len(thresholds),
    figsize=(6.8, 2.5),
    sharex=True,
    sharey=True
)

axes = axes.flatten()

colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

for threshold, ax in zip(thresholds, axes):

    for epc in [1, 2, 3, 4]:

        df_epc = df_occ[df_occ["EPceeb12e_efus"] == epc]
        epc_color = colors[4 - epc]

        tmp = df_epc.copy()

        # Threshold exceedance
        tmp["exceed"] = (
            tmp["T_in"] >= threshold
        ).astype(int)

        # Dwelling-day exceedance
        freq = (
            tmp.groupby(["CaseID", "date", "T_2DMMT"])
            .agg(
                exceed_hours=("exceed", "sum"),
                occupied_hours=("exceed", "count")
            )
            .reset_index()
        )

        freq["exceedance_pct"] = (
            100 * freq["exceed_hours"] / freq["occupied_hours"]
        )

        freq["overheating"] = (
            freq["exceedance_pct"] > 1
        )

        # Sort by 2DMMT
        freq = freq.sort_values("T_2DMMT")

        # Cumulative overheating frequency
        freq["cum_overheating"] = (
            100 * freq["overheating"].cumsum()
            # / np.arange(1, len(freq) + 1) # all dwellings days 
            / freq["overheating"].sum() 
        )

        ax.plot(
            freq["T_2DMMT"],
            freq["cum_overheating"],
            marker="o",
            markersize=0,
            linewidth=1.4,
            color=epc_color,
            label=EPC_AXIS_LABELS[epc].split(chr(10))[0]
        )

    ax.set_yscale('log')

    # Subplot formatting
    ax.set_title(
        f"{threshold}$^\\circ$C threshold",
        fontsize=8
    )

    ax.tick_params(axis="x", labelbottom=True)

    # ax.set_ylim(0, 100)

    ax.yaxis.set_major_formatter(
        mtick.PercentFormatter()
    )

    ax.grid(axis="y")

# Shared labels
fig.supylabel(
    "Cumulative distribution of 2DMMT\non fixed-threshold overheating days"
)

fig.supxlabel(
        "Outdoor Two-day Mean Max Temperature (2DMMT) ($^\\circ$C)"
    )


# Shared legend
handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=4,
    frameon=True,
    fontsize=6,
    title="EPC band",
    bbox_to_anchor=(0.5, -0.15)
)

fig.suptitle(
    f"Indoor {room_name} cumulative fixed threshold exceedance by EPC score band and 2DMMT\n"
    "EFUS 2017 London subset, May-September\n"
    "(CDF of 2DMMT among overheating days)",
    y=1.03
)

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/london_bedroom_4month/description/"
    f"fixed_cumulative_exceedance_vs_2dmmt_threshold_subplots_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Distribution of exceedance percentages by EPC band

import matplotlib.colors as mcolors

thresholds = [26, 27, 28]
# thresholds = [26, 27, 28]

colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

# fig, axes = plt.subplots(
#     1, len(thresholds),
#     figsize=(6.8, 2.6),
#     sharey=True
# )

fig, axes = plt.subplots(
    1, len(thresholds),
    figsize=(6.8, 2.5),
    sharex=True,
    sharey=True
)

for threshold, ax in zip(thresholds, axes.flatten()):

    exceedance_results = []

    for epc in display_bands:

        tmp = df_occ[df_occ["EPceeb12e_efus"] == epc].copy()

        tmp["exceed"] = (
            tmp["T_in"] >= threshold
        ).astype(int)

        freq = (
            tmp.groupby("CaseID")
            .agg(
                exceed_hours=("exceed", "sum"),
                occupied_hours=("exceed", "count")
            )
            .reset_index()
        )

        freq["exceedance_pct"] = (
            100 * freq["exceed_hours"] / freq["occupied_hours"]
        )

        exceedance_results.append(
            freq["exceedance_pct"].values
        )

    # Violin plot
    parts = ax.violinplot(
        exceedance_results,
        positions=range(1, 5),
        showmeans=False,
        showmedians=False,
        showextrema=True
    )

    dark_colors = []

    # Style violins
    for i, pc in enumerate(parts["bodies"]):

        color = colors[i]
        dark_color = tuple(np.array(mcolors.to_rgb(color)) * 0.45)

        dark_colors.append(dark_color)

        pc.set_facecolor(color)
        pc.set_edgecolor(color)
        pc.set_alpha(0.45)

    # Keep only central spine
    parts["cmins"].set_visible(False)
    parts["cmaxes"].set_visible(False)

    parts["cbars"].set_linewidth(1.2)
    parts["cbars"].set_color(dark_colors)

    # IQR + median styling
    for i, data in enumerate(exceedance_results, start=1):

        color = colors[i - 1]
        dark_color = dark_colors[i - 1]

        q1, med, q3 = np.percentile(data, [25, 50, 75])

        # IQR box
        ax.add_patch(
            plt.Rectangle(
                (i - 0.06, q1),
                0.12,
                q3 - q1,
                facecolor=color,
                edgecolor=dark_color,
                linewidth=1.2,
                zorder=4
            )
        )

        # Median line
        ax.plot(
            [i - 0.044, i + 0.044],
            [med, med],
            color="white",
            linewidth=1.2,
            solid_capstyle="butt",
            zorder=5
        )

        # Median label
        ax.text(
            i + 0.23,
            med,
            f"{med:.1f}%",
            fontsize=5,
            va="center",
            ha="center",
            color=dark_color
        )

       # IQR label at top of violin spine
        y_offset = np.max(data) + 2

        ax.text(
            i,
            y_offset,
            f"IQR\n[{q1:.1f}, {q3:.1f}]",
            fontsize=5,
            va="bottom",
            ha="center",
            color=dark_color
        )

    # Formatting
    # ymin = min(np.min(d) for d in exceedance_results)
    # ymax = max(np.max(d) for d in exceedance_results)

    ax.set_ylim([0, 100])

    ax.set_xticks(range(1, 5))

    ax.set_xticklabels([
        EPC_AXIS_LABELS[b] for b in display_bands
    ])

    ax.tick_params(
        axis="x",
        which="minor",
        bottom=False,
        top=False
    )

    

    ax.tick_params(axis="x", labelbottom=True)

    ax.set_title(
        f"{threshold}$^\\circ$C threshold",
        fontsize=8
    )

    ax.grid(axis="y")

    ax.yaxis.set_major_formatter(
        mtick.PercentFormatter()
    )

fig.supxlabel("EPC score band")

fig.supylabel(
    "Fraction of occupied hours above threshold"
)

fig.suptitle(
    f"Indoor {room_name} exceedance distributions by EPC band\n"
    "EFUS 2017 London subset, May-September",
    y=1.03
)

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/london_bedroom_4month/description/"
    f"fixed_exceedance_distribution_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Probability of overheating by EPC band
# Styled consistently with previous plots

import matplotlib.ticker as mtick
import matplotlib.colors as mcolors

fig, axes = plt.subplots(
    1, len(thresholds),
    figsize=(6.8, 2.6),
    sharey=True
)

for threshold, ax in zip(thresholds, axes):

    probs = []
    dark_colors = []

    overheating_counts = []
    total_counts = []

    for epc in display_bands:

        tmp = df_occ[df_occ["EPceeb12e_efus"] == epc].copy()

        tmp["exceed"] = (
            tmp["T_in"] >= threshold
        ).astype(int)

        freq = (
            tmp.groupby("CaseID")
            .agg(
                exceed_hours=("exceed", "sum"),
                occupied_hours=("exceed", "count")
            )
            .reset_index()
        )

        freq["exceedance_pct"] = (
            100 * freq["exceed_hours"] / freq["occupied_hours"]
        )

        overheating = (
            freq["exceedance_pct"] > 1
        )

        overheating_prob = overheating.mean()

        probs.append(100 * overheating_prob)

        overheating_counts.append(
            overheating.sum()
        )

        total_counts.append(
            len(overheating)
        )

        color = colors[len(probs) - 1]

        dark_colors.append(
            tuple(np.array(mcolors.to_rgb(color)) * 0.45)
        )

    # Bars
    bars = ax.bar(
        range(1, 5),
        probs,
        color=colors[:4],
        width=0.65,
        alpha=0.8,
        linewidth=1.2
    )

    # Bar styling + labels
    for i, (bar, prob) in enumerate(zip(bars, probs)):

        bar.set_edgecolor(dark_colors[i])

        n_overheat = overheating_counts[i]
        n_total = total_counts[i]

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            prob + 2,
            # f"{prob:.1f}%\n({n_overheat}/{n_total})",
            f"({n_overheat}/{n_total})",
            fontsize=5,
            ha="center",
            va="bottom",
            color=dark_colors[i]
        )

    # Formatting
    ax.set_title(
        f"{threshold}$^\\circ$C threshold",
        fontsize=8
    )

    ax.set_xticks(range(1, 5))

    ax.set_xticklabels([
        EPC_AXIS_LABELS[b]
        for b in display_bands
    ])

    ax.set_ylim(0, 110)

    ax.grid(axis="y")

    ax.yaxis.set_major_formatter(
        mtick.PercentFormatter()
    )

    ax.tick_params(
        axis="x",
        which="minor",
        bottom=False,
        top=False
    )

axes[0].set_ylabel(
    "Fraction of dwellings classified as overheating"
)

fig.supxlabel(
    "EPC score band"
)

fig.suptitle(
    f"Indoor {room_name} overheating probability by EPC band\n"
    r"Fixed threshold ($>1\%$ occupied hours)",
    y=1.03
)

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/london_bedroom_4month/description/"
    f"fixed_overheating_probability_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Degree-hours exceedance
# Styled consistently with previous violin plots

import matplotlib.colors as mcolors

fig, axes = plt.subplots(
    1, len(thresholds),
    figsize=(6.8, 2.6),
    sharey=True
)

for threshold, ax in zip(thresholds, axes):

    dh_results = []

    for epc in display_bands:

        tmp = df_occ[df_occ["EPceeb12e_efus"] == epc].copy()

        # Degree-hours above threshold
        tmp["degree_hours"] = (
            tmp["T_in"] - threshold
        ).clip(lower=0)

        dh = (
            tmp.groupby("CaseID")
            .agg(
                degree_hours=("degree_hours", "sum")
            )
            .reset_index()
        )

        dh_results.append(
            dh["degree_hours"].values
        )

    # Violin plot
    parts = ax.violinplot(
        dh_results,
        positions=range(1, 5),
        showmeans=False,
        showmedians=False,
        showextrema=True
    )

    dark_colors = []

    # Style violins
    for i, pc in enumerate(parts["bodies"]):

        color = colors[i]
        dark_color = tuple(np.array(mcolors.to_rgb(color)) * 0.45)

        dark_colors.append(dark_color)

        pc.set_facecolor(color)
        pc.set_edgecolor(color)
        pc.set_alpha(0.45)

    # Central spine only
    parts["cmins"].set_visible(False)
    parts["cmaxes"].set_visible(False)

    parts["cbars"].set_linewidth(1.2)
    parts["cbars"].set_color(dark_colors)

    # IQR + median styling
    for i, data in enumerate(dh_results, start=1):

        color = colors[i - 1]
        dark_color = dark_colors[i - 1]

        q1, med, q3 = np.percentile(data, [25, 50, 75])

        # IQR box
        ax.add_patch(
            plt.Rectangle(
                (i - 0.06, q1),
                0.12,
                q3 - q1,
                facecolor=color,
                edgecolor=dark_color,
                linewidth=1.2,
                zorder=4
            )
        )

        # Median line
        ax.plot(
            [i - 0.049, i + 0.049],
            [med, med],
            color="white",
            linewidth=1.2,
            solid_capstyle="butt",
            zorder=5
        )

        # Median label
        ax.text(
            i + 0.23,
            med,
            f"{med:.1f}",
            fontsize=5,
            va="center",
            ha="center",
            color=dark_color
        )

        # IQR label at top of violin spine
        y_offset = np.max(data) + 2

        ax.text(
            i,
            y_offset,
            f"IQR\n[{q1:.1f}, {q3:.1f}]",
            fontsize=5,
            va="bottom",
            ha="center",
            color=dark_color
        )

    # Formatting
    ymin = min(np.min(d) for d in dh_results)
    ymax = max(np.max(d) for d in dh_results)

    ax.set_ylim([
        0,2000
    ])

    ax.set_xticks(range(1, 5))

    ax.set_xticklabels([
        EPC_AXIS_LABELS[b]
        for b in display_bands
    ])

    ax.tick_params(
        axis="x",
        which="minor",
        bottom=False,
        top=False
    )

    ax.set_title(
        f"{threshold}$^\\circ$C threshold",
        fontsize=8
    )

    ax.grid(axis="y")

axes[0].set_ylabel(
    "Degree-hours above threshold"
)

fig.supxlabel(
    "EPC score band"
)

fig.suptitle(
    f"Indoor {room_name} overheating severity by EPC band\n"
    "Degree-hours exceedance",
    y=1.03
)

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/london_bedroom_4month/description/"
    f"fixed_degree_hours_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Duration analysis: longest continuous overheating event
# Styled consistently with previous violin plots

import matplotlib.colors as mcolors

fig, axes = plt.subplots(
    1, len(thresholds),
    figsize=(6.8, 2.6),
    sharey=True
)

# Store results for printing later
all_duration_results = {}

for threshold, ax in zip(thresholds, axes):

    duration_results = []

    # Create nested dict for this threshold
    all_duration_results[threshold] = {}

    for epc in display_bands:

        tmp = (
            df_occ[df_occ["EPceeb12e_efus"] == epc]
            .sort_values(["CaseID", "hour"])
            .copy()
        )

        tmp["exceed"] = (
            tmp["T_in"] >= threshold
        ).astype(int)

        durations = []

        # Store CaseID -> longest duration
        duration_by_house = {}

        for case_id, group in tmp.groupby("CaseID"):

            runs = (
                group["exceed"]
                .groupby(
                    (
                        group["exceed"]
                        != group["exceed"].shift()
                    ).cumsum()
                )
                .cumsum()
            )

            max_duration = runs.max()

            durations.append(max_duration)

            duration_by_house[case_id] = max_duration

        # Save for later printing
        all_duration_results[threshold][epc] = duration_by_house

        duration_results.append(
            np.array(durations)
        )

    # Violin plot
    parts = ax.violinplot(
        duration_results,
        positions=range(1, 5),
        showmeans=False,
        showmedians=False,
        showextrema=True
    )

    dark_colors = []

    # Style violins
    for i, pc in enumerate(parts["bodies"]):

        color = colors[i]

        dark_color = tuple(
            np.array(mcolors.to_rgb(color)) * 0.45
        )

        dark_colors.append(dark_color)

        pc.set_facecolor(color)
        pc.set_edgecolor(color)
        pc.set_alpha(0.45)

    # Keep only central spine
    parts["cmins"].set_visible(False)
    parts["cmaxes"].set_visible(False)

    parts["cbars"].set_linewidth(1.2)
    parts["cbars"].set_color(dark_colors)

    # IQR + median styling
    for i, data in enumerate(duration_results, start=1):

        color = colors[i - 1]
        dark_color = dark_colors[i - 1]

        q1, med, q3 = np.percentile(
            data,
            [25, 50, 75]
        )

        # IQR box
        ax.add_patch(
            plt.Rectangle(
                (i - 0.06, q1),
                0.12,
                q3 - q1,
                facecolor=color,
                edgecolor=dark_color,
                linewidth=1.2,
                zorder=4
            )
        )

        # Median line
        ax.plot(
            [i - 0.044, i + 0.044],
            [med, med],
            color="white",
            linewidth=1.2,
            solid_capstyle="butt",
            zorder=5
        )

        # Median label
        ax.text(
            i + 0.3,
            med,
            f"{med:.1f} h",
            fontsize=5,
            va="center",
            ha="center",
            color=dark_color
        )

        # IQR label
        y_offset = np.max(data) + 1

        ax.text(
            i,
            y_offset,
            f"IQR\n[{q1:.1f}, {q3:.1f}]",
            fontsize=5,
            va="bottom",
            ha="center",
            color=dark_color
        )

    # Formatting
    ax.set_ylim([
        0, 500
    ])

    ax.set_xticks(range(1, 5))

    ax.set_xticklabels([
        EPC_AXIS_LABELS[b]
        for b in display_bands
    ])

    ax.tick_params(
        axis="x",
        which="minor",
        bottom=False,
        top=False
    )

    ax.set_title(
        f"{threshold}$^\\circ$C threshold",
        fontsize=8
    )

    ax.grid(axis="y")

axes[0].set_ylabel(
    "Longest continuous overheating event (hours)"
)

fig.supxlabel(
    "EPC score band"
)

fig.suptitle(
    f"Indoor {room_name} overheating duration by EPC band\n"
    "Longest continuous overheating event",
    y=1.03
)

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/london_bedroom_4month/description/"
    f"fixed_duration_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()

print("\nLongest overheating events by EPC band")

for threshold in thresholds:

    print("\n" + "=" * 60)
    print(f"Threshold: {threshold}°C")
    print("=" * 60)

    for epc in display_bands:

        duration_by_house = all_duration_results[threshold][epc]

        sorted_houses = sorted(
            duration_by_house.items(),
            key=lambda x: x[1],
            reverse=True
        )

        print(f"\nEPC {epc}")
        print("-" * 30)

        for case_id, duration in sorted_houses:

            print(
                f"CaseID {case_id}: "
                f"{duration:.0f} hours"
            )

In [ ]:
plot_temps(["2230","2042","2422","2469"])

In [ ]:
# Get all post-1990 dwellings
post_1990_ids = (
    dwelling_chars[
        dwelling_chars["dwage_efus"] == 7
    ]["CaseID"]
    .astype(str)
    .unique()
    .tolist()
)

print(
    f"Number of post-1990 dwellings: "
    f"{len(post_1990_ids)}"
)

# Plot all post-1990 dwellings
plot_temps(post_1990_ids)

In [ ]:
# fixed threshold exceedance envelope vs 2DMMT by EPC band
# Filled region spans min–max exceedance across thresholds

import matplotlib.ticker as mtick

fig, ax = plt.subplots(figsize=(3.4, 2.6))

colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

for epc in [1, 2, 3, 4]:

    df_epc = df_occ[df_occ["EPceeb12e_efus"] == epc]
    epc_color = colors[4 - epc]

    threshold_props = []

    # Calculate exceedance curves for all thresholds
    for threshold in thresholds:

        tmp = df_epc.copy()

        tmp["exceed"] = (
            tmp["T_in"] >= threshold
        ).astype(int)

        freq = (
            tmp.groupby(["CaseID", "date", "T_2DMMT"])
            .agg(
                exceed_hours=("exceed", "sum"),
                occupied_hours=("exceed", "count")
            )
            .reset_index()
        )

        freq["exceedance_pct"] = (
            100 * freq["exceed_hours"] / freq["occupied_hours"]
        )

        freq["overheating"] = (
            freq["exceedance_pct"] > 1
        )

        freq["T_2DMMT_bin"] = (
            freq["T_2DMMT"].round()
        )

        prop = (
            freq.groupby("T_2DMMT_bin")["overheating"]
            .mean()
            .reset_index()
        )

        prop["overheating"] *= 100

        threshold_props.append(
            prop.set_index("T_2DMMT_bin")["overheating"]
        )

    # Combine thresholds into one dataframe
    prop_df = pd.concat(threshold_props, axis=1)

    x = prop_df.index.values
    y_min = prop_df.min(axis=1).values
    y_max = prop_df.max(axis=1).values
    y_mid = prop_df.mean(axis=1).values

    # Filled range
    ax.fill_between(
        x,
        y_min,
        y_max,
        color=epc_color,
        alpha=0.2,
        label=EPC_AXIS_LABELS[epc].split(chr(10))[0]
    )

    # Mean line
    ax.plot(
        x,
        y_mid,
        color=epc_color,
        linewidth=1.5
    )

# Formatting
ax.set_xlabel("Outdoor Two-day Mean Max Temperature (2DMMT) ($^\\circ$C)")

ax.set_ylabel(
    r"Dwellings exceeding fixed threshold (\%)"
)

ax.set_ylim(0, 100)
ax.set_xlim(min(x),max(x))

ax.set_title(
    f"Indoor {room_name} fixed threshold exceedance by EPC score band and 2DMMT\n"
    "EFUS 2017 London subset, May-September"
)

ax.yaxis.set_major_formatter(mtick.PercentFormatter())

ax.grid(axis="y")

ax.legend(
    fontsize=5,
    frameon=False,
    loc='upper left'
)

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/london_bedroom_4month/description/"
    f"fixed_exceedance_vs_2dmmt_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()